# Data Analysis - Statistical Inference

### 0. Import Modules and Load Data

In [ ]:
import pandas as pd
import numpy as np
import os 
from pathlib import Path
import re
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from itertools import combinations
import statsmodels.api as sm
import statsmodels.formula.api as smf
import statsmodels.stats.api as sms
import statsmodels.stats.multicomp as multi
import statsmodels.stats.anova as anova
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures
from matplotlib.patches import Patch 
from scipy import stats
from scipy.stats import pearsonr, spearmanr
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import pingouin as pg

os.chdir("C:\\Users\\ThinkPad\\Desktop\\llm_biases_fse\\00_data\\final_data")
# Load Data
df_llama = pd.read_csv('llama_final_data.csv')
df_qwen = pd.read_csv('qwen_final_data.csv')
df_mistral = pd.read_csv('mistral_final_data.csv')
vignette_cols = ['race', 'gender', 'religion', 'gender_alignment']  # or 'transness'
os.chdir("C:\\Users\\ThinkPad\\Desktop\\llm_biases_fse\\00_data")
vignette_complete = pd.read_csv('complete_vignette_llm.csv')

os.chdir("C:\\Users\\ThinkPad\\Desktop\\llm_biases_fse\\03_classical_statistics")
import myfunctions as mf

In [ ]:
from importlib import reload
mf = reload(mf)


In [ ]:
# Merge vignette_complete into all DataFrame variables whose name starts with "df_"
# - Adds "vignette_" prefix to vignette_complete.vignette_id when missing
# - Produces new DataFrames named <orig_name>_merged (does not overwrite originals)

vc = vignette_complete.copy()
vc['vignette_id'] = vc['vignette_id'].astype(str).str.strip()
vc['vignette_id'] = vc['vignette_id'].where(vc['vignette_id'].str.startswith('vignette_'),
                                            'vignette_' + vc['vignette_id'])

merged_vars = []
for name, obj in list(globals().items()):
    if name.startswith("df_") and isinstance(obj, pd.DataFrame):
        try:
            merged = obj.merge(vc, on="vignette_id", how="left", suffixes=("", "_vign"))
            out_name = f"{name}_merged"
            globals()[out_name] = merged
            merged_vars.append(out_name)
            print(f"Created {out_name}: {merged.shape[0]} rows x {merged.shape[1]} cols")
        except Exception as e:
            print(f"Failed to merge {name}: {e}")

print(f"\nTotal merged dataframes created: {len(merged_vars)}")

# Global, aggregated (over all concepts) Models for all LLMs

### 1. Global aggregated Llama

##### 1.1. Recoding of Response Variable, Generation of Interaction Columns

In [ ]:
df_llama_collapsed = mf.recode_response_variable(df_llama)
df_llama_collapsed = mf.collapse_vignette_responses(df_llama_collapsed, id_cols=("item_id", "vignette_id"), response_cols =("response", "response_recoded"), metadata_strategy="first")
df = df_llama_collapsed.dropna(subset=["response_recoded"]).copy()
df = df.loc[:, ~df.columns.str.endswith(('_cat', '_clean'))]
# Recode gender value nonbinary person to nonbinary
df['gender'] = df['gender'].replace({'nonbinary person': 'nonbinary'})
df['gender'] = df['gender'].replace({'person': 'not_mentioned'})
# rename column gender_alignment to transness
df = df.rename(columns={'gender_alignment': 'transness'})

vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)

print(f"Total dataset size: {len(df):,} observations")
print(f"Number of concepts: {df['concept'].nunique()}")
print(f"Concepts: {df['concept'].unique()}")

plot_dir = "maihda_mixed_reg_plots_llama"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_dataset_mixed_model_results_llama.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")




##### 1.2. MAIHDA


In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_llama_ready = df.copy()
res_llama = mf.run_maihda_simple(
    df,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    out_dir="maihda_full_llama",
    reml=True,
    optimizer="powell",
    disp=False
)

### 2. Global aggregated Qwen

##### 2.1. Recoding of Response Variable, Generation of Interaction Columns

In [ ]:
df_qwen_collapsed = mf.recode_response_variable(df_qwen)
df_qwen_collapsed = mf.collapse_vignette_responses(df_qwen_collapsed, id_cols=("item_id", "vignette_id"), response_cols =("response", "response_recoded"), metadata_strategy="first")
df = df_qwen_collapsed.dropna(subset=["response_recoded"]).copy()
df = df.loc[:, ~df.columns.str.endswith(('_cat', '_clean'))]
# Recode gender value nonbinary person to nonbinary
df['gender'] = df['gender'].replace({'nonbinary person': 'nonbinary'})
df['gender'] = df['gender'].replace({'person': 'not_mentioned'})
# rename column gender_alignment to transness
df = df.rename(columns={'gender_alignment': 'transness'})

vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)

print(f"Total dataset size: {len(df):,} observations")
print(f"Number of concepts: {df['concept'].nunique()}")
print(f"Concepts: {df['concept'].unique()}")

plot_dir = "maihda_mixed_reg_plots_qwen"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_dataset_mixed_model_results_qwen.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")

##### 2.2. MAIHDA Models

In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_qwen_ready = df.copy()
res_qwen = mf.run_maihda_simple(
    df,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    out_dir="maihda_full_qwen",
    reml=True,
    optimizer="powell",
    disp=False
)

### 3. Global aggregated Mistral

##### 3.1. Preparations

In [ ]:
df_mistral_collapsed = mf.recode_response_variable(df_mistral)
df_mistral_collapsed = mf.collapse_vignette_responses(df_mistral_collapsed, id_cols=("item_id", "vignette_id"), response_cols =("response", "response_recoded"), metadata_strategy="first")
df = df_mistral_collapsed.dropna(subset=["response_recoded"]).copy()
df = df.loc[:, ~df.columns.str.endswith(('_cat', '_clean'))]
# Recode gender value nonbinary person to nonbinary
df['gender'] = df['gender'].replace({'nonbinary person': 'nonbinary'})
df['gender'] = df['gender'].replace({'person': 'not_mentioned'})
# rename column gender_alignment to transness
df = df.rename(columns={'gender_alignment': 'transness'})

vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)

print(f"Total dataset size: {len(df):,} observations")
print(f"Number of concepts: {df['concept'].nunique()}")
print(f"Concepts: {df['concept'].unique()}")

plot_dir = "maihda_mixed_reg_plots_mistral"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_dataset_mixed_model_results_mistral.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")

##### 3.2. MAIHDA Models

In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_mistral_ready = df.copy()
res_mistral = mf.run_maihda_simple(
    df,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    out_dir="maihda_full_mistral",
    reml=True,
    optimizer="powell",
    disp=False
)

## OLS models for all LLMs

In [ ]:
# One OLS per LLM: use available processed dataframes (prefer variables defined in later cells)

def _pick_df(*names):
    for n in names:
        if n in globals() and isinstance(globals()[n], pd.DataFrame):
            return globals()[n]
    return None

datasets = {
    "llama": _pick_df("df_llama_ready", "df_llama_collapsed", "df_llama"),
    "qwen":  _pick_df("df_qwen_ready", "df_qwen_collapsed", "df_qwen"),
    "mistral": _pick_df("df_mistral_ready", "df_mistral_collapsed", "df_mistral")
}

out_dir = Path("ols_results")
out_dir.mkdir(exist_ok=True)

formula = "response_recoded ~ C(gender) + C(race) + C(religion) + C(transness) + C(concept)"

for name, df_model in datasets.items():
    if df_model is None:
        print(f"Skipping {name}: no suitable dataframe found.")
        continue

    df_use = df_model.dropna(subset=["response_recoded"]).copy()
    if df_use.empty:
        print(f"Skipping {name}: dataframe is empty after dropping NA response_recoded.")
        continue

    try:
        mdl = smf.ols(formula=formula, data=df_use).fit()
    except Exception as e:
        print(f"Failed to fit OLS for {name}: {e}")
        continue

    ci = mdl.conf_int()
    res_df = pd.DataFrame({
        "coef": mdl.params,
        "std_err": mdl.bse,
        "t": mdl.tvalues,
        "pvalue": mdl.pvalues,
        "ci_lower": ci.iloc[:,0],
        "ci_upper": ci.iloc[:,1]
    })
    out_csv = out_dir / f"ols_results_{name}.csv"
    res_df.to_csv(out_csv)
    # also save model summary text for quick inspection
    (out_dir / f"ols_summary_{name}.txt").write_text(mdl.summary().as_text())

    print(f"Saved OLS results for {name} -> {out_csv} ({res_df.shape[0]} coefficients)")

In [ ]:

plot_dir = Path("ols_coef_plots")
plot_dir.mkdir(exist_ok=True)

csv_files = list(out_dir.glob("ols_results_*.csv"))
if not csv_files:
    raise FileNotFoundError(f"No OLS CSVs found in {out_dir!s}")

def pretty_label(s):
    s = str(s)
    s = s.replace("Intercept", "Intercept")
    s = re.sub(r"C\(([^)]+)\)\[T\.([^\]]+)\]", r"\1: \2", s)        # C(var)[T.level] -> var: level
    s = s.replace("C(", "").replace(")", "")
    #s = s.replace(":", " — ")
    s = s.replace("T.", "")
    s = s.replace(" ", "")
    s = s.replace("Cgender", "gender")
    s = s.replace("Crace", "race")
    s = s.replace("Creligion", "religion")
    s = s.replace("Ctransness", "transness")
    s = s.replace("Cconcept", "concept")
    return s

for csv_path in csv_files:
    df = pd.read_csv(csv_path, index_col=0)
    for col in ("coef", "ci_lower", "ci_upper", "pvalue"):
        if col not in df.columns:
            raise KeyError(f"Expected column '{col}' in {csv_path.name}")

    df_plot = df.copy()
    # omit concept coefficients
    df_plot = df_plot[~df_plot.index.str.startswith("C(concept)")].copy()
    df_plot = df_plot[~df_plot.index.str.contains('Intercept|const', case=False)]
    if df_plot.empty:
        print(f"Skipping {csv_path.name}: no coefficients left after removing concepts.")
        continue

    df_plot["name"] = df_plot.index.astype(str).map(pretty_label)
    df_plot = df_plot.sort_values("coef")
    y = range(len(df_plot))
    coef = df_plot["coef"].values
    ci_lower = df_plot["ci_lower"].values
    ci_upper = df_plot["ci_upper"].values
    colors = ["#7d0a04" if p < 0.05 else "#5b6d85" for p in df_plot["pvalue"].fillna(1.0).values]

    fig, ax = plt.subplots(figsize=(8, max(4, len(df_plot) * 0.25)))

    # Thin, subtle CI lines
    ax.hlines(y=y, xmin=ci_lower, xmax=ci_upper, color='gray', alpha=0.4, linewidth=1.5)

    # Prominent coefficient points colored by significance
    ax.scatter(coef, y, c=colors, edgecolor="black", zorder=3, s=70, linewidth=0.5, alpha=0.9)

    ax.axvline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_yticks(y)
    ax.set_yticklabels(df_plot["name"].tolist(), fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel("Coefficient (95% CI)", fontsize=10)
    ax.grid(axis='x', alpha=0.2, linestyle=':')
    # zoom to CI range with a small padding so CIs are clearly visible
    x_min = ci_lower.min()
    x_max = ci_upper.max()
    pad = max((x_max - x_min) * 0.1, 0.05)
    ax.set_xlim(x_min - pad, x_max + pad)

    model_name = csv_path.stem.replace("ols_results_", "")
    ax.set_title(f"OLS coefficients — {model_name.upper()}")
    plt.tight_layout()

    out_file = plot_dir / f"ols_coefs_{model_name}.png"
    plt.savefig(out_file, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved {out_file} ({len(df_plot)} coefficients)")

## Bootstrap Overall OLS

In [ ]:
# Assumes cluster_bootstrap_ols_minimal is already defined

formula = "response_recoded ~ C(gender) + C(race) + C(religion) + C(transness) + C(concept)"
cluster_col = "vignette_id"
B = 2000
alpha = 0.05

datasets = {
    "llama": df_llama_ready,
    "qwen": df_qwen_ready,
    "mistral": df_mistral_ready,
}

out_dir = Path("bootstrap_ols_exports")
out_dir.mkdir(exist_ok=True)

results = []

for llm, df in datasets.items():
    print(f"Bootstrapping {llm}...")

    boot = mf.cluster_bootstrap_ols_minimal(
        formula=formula,
        df=df,
        cluster_col=cluster_col,
        B=B,
        seed=42
    ).copy()

    boot.insert(0, "term", boot.index)
    boot.insert(0, "llm", llm)
    boot["significant"] = boot["p_boot"] < alpha

    results.append(boot.reset_index(drop=True))

# Long-format results (all coefficients, all LLMs)
all_long = pd.concat(results, ignore_index=True)



### OLS Masterplot

In [ ]:
mf = reload(mf)

In [ ]:


drop_terms = ["Intercept", "ADM", "CON", "ENV", "PIT", "AH", "PH", "AF", "PF"]
df_filt = all_long[~all_long["term_simple"].isin(drop_terms)].copy()
llm_labels = {
    "llama": "LLaMA",
    "mistral": "Mistral",
    "qwen": "Qwen"
}

term_labels = {
    "man": "man",
    "woman": "woman",
    "nonbinary": "non-binary",
    "christian": "Christian",
    "muslim": "Muslim",
    "jewish": "Jewish",
    "black": "Black",
    "white": "White",
    "asian": "Asian",
    "trans": "trans",
    "cis" : "cis"
}
term_order = [
    "man",
    "woman",
    "nonbinary",
    "trans",
    "cis",
    "black",
    "white",
    "asian",
    "christian",
    "muslim",
    "jewish",
]
df_plot = df_filt.copy()
df_plot["llm_label"] = df_plot["llm"].map(llm_labels).fillna(df_plot["llm"])
df_plot["term_label"] = df_plot["term_simple"].map(term_labels).fillna(df_plot["term_simple"])
df_plot["term_label"] = pd.Categorical(
    df_plot["term_label"],
    categories=[term_labels[t] for t in term_order],
    ordered=True
)

fig, ax = mf.plot_ols_coefs_forest(
    df_plot,
    term_col="term_label",
    llm_col="llm_label",
    sort_by="none",
    font_scale=2,
    title="OLS coefficients with 95% bootstrapped CIs"
)
plt.show()


In [ ]:
wide = all_long.pivot_table(
    index="term",
    columns="llm",
    values=["coef", "ci_low", "ci_high", "p_boot"],
    aggfunc="first"
)

# Flatten column names: coef_llama, ci_low_qwen, etc.
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# Keep terms significant in at least one LLM
sig_any = np.zeros(len(wide), dtype=bool)
for llm in datasets.keys():
    pcol = f"p_boot_{llm}"
    if pcol in wide.columns:
        sig_any |= (wide[pcol] < alpha)

wide_sig = wide.loc[sig_any].copy()

csv_sig = out_dir / "bootstrap_ols_significant_terms_wide.csv"
wide_sig.to_csv(csv_sig, index=False)
print(f"Saved: {csv_sig}")

# Add simplified label + stars to the long table
all_long = all_long.copy()
all_long["term_simple"] = all_long["term"].apply(lambda s: mf.simplify_term(s, keep_prefix=False))
all_long["stars"] = all_long["p_boot"].apply(mf.stars)

csv_all_pretty = out_dir / "bootstrap_ols_all_llms_long_pretty.csv"
all_long.to_csv(csv_all_pretty, index=False)
print(f"Saved: {csv_all_pretty}")

wide_sig = wide_sig.copy()
wide_sig["term_simple"] = wide_sig["term"].apply(lambda s: mf.simplify_term(s, keep_prefix=False))

# Add stars per LLM based on p_boot_<llm>
for llm in ["llama", "qwen", "mistral"]:
    pcol = f"p_boot_{llm}"
    if pcol in wide_sig.columns:
        wide_sig[f"stars_{llm}"] = wide_sig[pcol].apply(mf.stars)

# (Optional) move term_simple next to term for readability
cols = ["term", "term_simple"] + [c for c in wide_sig.columns if c not in ("term", "term_simple")]
wide_sig = wide_sig[cols]

csv_sig_pretty = out_dir / "bootstrap_ols_significant_terms_wide_pretty.csv"
wide_sig.to_csv(csv_sig_pretty, index=False)
print(f"Saved: {csv_sig_pretty}")



# Warmth-Competence-Scores

## Llama Score and Models

In [ ]:
df_llama_vignette = mf.compute_scm_scores(
        df_llama_ready, 
        response_col='response',
        concept_col='concept',
        groupby_col='vignette_id',
        keep_cols=['gender', 'race', 'religion', 'transness']
    )

In [ ]:
from statsmodels.multivariate.manova import MANOVA

# convert predictors to categorical
for col in ["gender", "race", "religion", "transness"]:
    df_llama_vignette[col] = df_llama_vignette[col].astype("category")

formula = "warmth_score + competence_score ~ gender + race + religion + transness"

mv = MANOVA.from_formula(formula, data=df_llama_vignette)
print(mv.mv_test())
import statsmodels.formula.api as smf

model_w_ols_llama = smf.ols(
    "warmth_score ~ gender + race + religion + transness",
    data=df_llama_vignette
).fit()

model_c_ols_llama = smf.ols(
    "competence_score ~ gender + race + religion + transness",
    data=df_llama_vignette
).fit()

# Warmth coefficients → DataFrame
coef_w = model_w_ols_llama.params.rename("warmth_coef").to_frame()
# Competence coefficients → DataFrame
coef_c = model_c_ols_llama.params.rename("competence_coef").to_frame()
# Merge into combined coef table
coef_llama = coef_w.join(coef_c).drop(index="Intercept")
def family(term):
    if term.startswith("gender"): return "gender"
    if term.startswith("race"): return "race"
    if term.startswith("religion"): return "religion"
    if term.startswith("transness"): return "transness"
    return "other"

coef_llama["family"] = coef_llama.index.map(family)
se_w = model_w_ols_llama.bse.rename("se_w")
se_c = model_c_ols_llama.bse.rename("se_c")
se = se_w.to_frame().join(se_c.to_frame()).drop(index="Intercept")

cov_llama = {
    term: np.array([
        [se.loc[term, "se_w"]**2, 0],
        [0, se.loc[term, "se_c"]**2]
    ])
    for term in coef_llama.index
}

print(model_w_ols_llama.summary())
print(model_c_ols_llama.summary())

In [ ]:
df = df_llama_vignette.copy()
vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)


plot_dir = "maihda_warmth_llama"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_warmth_llama.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")

In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_llama_wc = df.copy()
res_warmth_llama_maihda = mf.run_maihda_simple(
    df_llama_wc,
    response_var="warmth_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_warmth_llama",
    reml=True,
    optimizer="powell",
    disp=False
)

res_comp_llama_maihda = mf.run_maihda_simple(
    df_llama_wc,
    response_var="competence_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_competence_llama",
    reml=True,
    optimizer="powell",
    disp=False
)

## Qwen Score and Models

In [ ]:
df_qwen_vignette = mf.compute_scm_scores(
        df_qwen_ready, 
        response_col='response',
        concept_col='concept',
        groupby_col='vignette_id',
        keep_cols=['gender', 'race', 'religion', 'transness']
    )

In [ ]:
from statsmodels.multivariate.manova import MANOVA

# convert predictors to categorical
for col in ["gender", "race", "religion", "transness"]:
    df_qwen_vignette[col] = df_qwen_vignette[col].astype("category")

formula = "warmth_score + competence_score ~ gender + race + religion + transness"

mv = MANOVA.from_formula(formula, data=df_qwen_vignette)
print(mv.mv_test())
import statsmodels.formula.api as smf

model_w_ols_qwen = smf.ols(
    "warmth_score ~ gender + race + religion + transness",
    data=df_qwen_vignette
).fit()

model_c_ols_qwen = smf.ols(
    "competence_score ~ gender + race + religion + transness",
    data=df_qwen_vignette
).fit()


# Warmth coefficients → DataFrame
coef_w = model_w_ols_qwen.params.rename("warmth_coef").to_frame()
# Competence coefficients → DataFrame
coef_c = model_c_ols_qwen.params.rename("competence_coef").to_frame()
# Merge into combined coef table
coef_qwen = coef_w.join(coef_c).drop(index="Intercept")
def family(term):
    if term.startswith("gender"): return "gender"
    if term.startswith("race"): return "race"
    if term.startswith("religion"): return "religion"
    if term.startswith("transness"): return "transness"
    return "other"

coef_qwen["family"] = coef_qwen.index.map(family)
se_w = model_w_ols_qwen.bse.rename("se_w")
se_c = model_c_ols_qwen.bse.rename("se_c")
se = se_w.to_frame().join(se_c.to_frame()).drop(index="Intercept")

cov_qwen = {
    term: np.array([
        [se.loc[term, "se_w"]**2, 0],
        [0, se.loc[term, "se_c"]**2]
    ])
    for term in coef_qwen.index
}


print(model_w_ols_qwen.summary())
print(model_c_ols_qwen.summary())

In [ ]:
df = df_qwen_vignette.copy()
vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)


plot_dir = "maihda_warmth_qwen"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_warmth_qwen.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")

In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_qwen_wc = df.copy()
res_warmth_qwen_maihda = mf.run_maihda_simple(
    df_qwen_wc,
    response_var="warmth_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_warmth_qwen",
    reml=True,
    optimizer="powell",
    disp=False
)

res_comp_qwen_maihda = mf.run_maihda_simple(
    df_qwen_wc,
    response_var="competence_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_competence_qwen",
    reml=True,
    optimizer="powell",
    disp=False
)

## Mistral Score and Models

In [ ]:
df_mistral_vignette = mf.compute_scm_scores(
        df_mistral_ready, 
        response_col='response',
        concept_col='concept',
        groupby_col='vignette_id',
        keep_cols  =['gender', 'race', 'religion', 'transness']
    )

In [ ]:
from statsmodels.multivariate.manova import MANOVA

# convert predictors to categorical
for col in ["gender", "race", "religion", "transness"]:
    df_mistral_vignette[col] = df_mistral_vignette[col].astype("category")

formula = "warmth_score + competence_score ~ gender + race + religion + transness"

mv = MANOVA.from_formula(formula, data=df_mistral_vignette)
print(mv.mv_test())
import statsmodels.formula.api as smf

model_w_ols_mistral = smf.ols(
    "warmth_score ~ gender + race + religion + transness",
    data=df_mistral_vignette
).fit()

model_c_ols_mistral = smf.ols(
    "competence_score ~ gender + race + religion + transness",
    data=df_mistral_vignette
).fit()

# Warmth coefficients → DataFrame
coef_w = model_w_ols_mistral.params.rename("warmth_coef").to_frame()
# Competence coefficients → DataFrame
coef_c = model_c_ols_mistral.params.rename("competence_coef").to_frame()
# Merge into combined coef table
coef_mistral = coef_w.join(coef_c).drop(index="Intercept")
def family(term):
    if term.startswith("gender"): return "gender"
    if term.startswith("race"): return "race"
    if term.startswith("religion"): return "religion"
    if term.startswith("transness"): return "transness"
    return "other"

coef_mistral["family"] = coef_mistral.index.map(family)
se_w = model_w_ols_mistral.bse.rename("se_w")
se_c = model_c_ols_mistral.bse.rename("se_c")
se = se_w.to_frame().join(se_c.to_frame()).drop(index="Intercept")

cov_mistral = {
    term: np.array([
        [se.loc[term, "se_w"]**2, 0],
        [0, se.loc[term, "se_c"]**2]
    ])
    for term in coef_mistral.index
}

print(model_w_ols_mistral.summary())
print(model_c_ols_mistral.summary())

In [ ]:
df = df_mistral_vignette.copy()
vignette_cols = ['race','gender','religion','transness']

# "not_mentioned" als Referenzlevel setzen
for col in vignette_cols:
    levels = df[col].dropna().unique().tolist()
    levels = ["not_mentioned"] + [x for x in levels if x != "not_mentioned"]
    df[col] = pd.Categorical(df[col].fillna("not_mentioned"), categories=levels, ordered=False)


plot_dir = "maihda_warmth_mistral"
os.makedirs(plot_dir, exist_ok=True)
out_fname = "maihda_warmth_mistral.xlsx"

# ------------------------
# Interaktionen (mit EXCLUDE für not_mentioned)
# ------------------------
interaction_cols = []
for a, b in itertools.combinations(vignette_cols, 2):
    col_name = f"{a}_{b}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str),
        "EXCLUDE"
    )
    interaction_cols.append(col_name)

print(f"Created {len(interaction_cols)} 2-way interaction columns (EXCLUDE marks invalid combos)")

interaction_cols_3way = []
for a, b, c in itertools.combinations(vignette_cols, 3):
    col_name = f"{a}_{b}_{c}"
    df[col_name] = np.where(
        (df[a] != "not_mentioned") & (df[b] != "not_mentioned") & (df[c] != "not_mentioned"),
        df[a].astype(str) + "_" + df[b].astype(str) + "_" + df[c].astype(str),
        "EXCLUDE"
    )
    interaction_cols_3way.append(col_name)

print(f"Created {len(interaction_cols_3way)} 3-way interaction columns (EXCLUDE marks invalid combos)")


In [ ]:
df["stratum"] = df[vignette_cols].astype(str).agg("_".join, axis=1)
print(f"✅ Created stratum variable with {df['stratum'].nunique()} unique strata.")
df_mistral_wc = df.copy()
res_warmth_mistral_maihda = mf.run_maihda_simple(
    df_mistral_wc,
    response_var="warmth_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_warmth_mistral",
    reml=True,
    optimizer="powell",
    disp=False
)

res_comp_mistral_maihda = mf.run_maihda_simple(
    df_mistral_wc,
    response_var="competence_score",
    fixed_effects=("race","gender","religion","transness"),
    stratum_col="stratum",
    out_dir="maihda_competence_mistral",
    reml=True,
    optimizer="powell",
    disp=False
)

## Bootstrapping

In [ ]:

formula = "warmth_score ~ C(gender) + C(race) + C(religion) + C(transness)"
cluster_col = "vignette_id"
B = 2000
alpha = 0.05

datasets = {
    "llama": df_llama_wc,
    "qwen": df_qwen_wc,
    "mistral": df_mistral_wc,
}

out_dir = Path("bootstrap_ols_warmth_score_exports")
out_dir.mkdir(exist_ok=True)

results = []

for llm, df in datasets.items():
    print(f"Bootstrapping {llm}...")

    boot = mf.cluster_bootstrap_ols_minimal(
        formula=formula,
        df=df,
        cluster_col=cluster_col,
        B=B,
        seed=42
    ).copy()

    boot.insert(0, "term", boot.index)
    boot.insert(0, "llm", llm)
    boot["significant"] = boot["p_boot"] < alpha

    results.append(boot.reset_index(drop=True))

# Long-format results (all coefficients, all LLMs)
all_long = pd.concat(results, ignore_index=True)



In [ ]:
wide = all_long.pivot_table(
    index="term",
    columns="llm",
    values=["coef", "ci_low", "ci_high", "p_boot"],
    aggfunc="first"
)

# Flatten column names: coef_llama, ci_low_qwen, etc.
wide.columns = [f"{a}_{b}" for a, b in wide.columns]
wide = wide.reset_index()

# Keep terms significant in at least one LLM
sig_any = np.zeros(len(wide), dtype=bool)
for llm in datasets.keys():
    pcol = f"p_boot_{llm}"
    if pcol in wide.columns:
        sig_any |= (wide[pcol] < alpha)

wide_sig = wide.loc[sig_any].copy()

csv_sig = out_dir / "bootstrap_ols_warmth_score_significant_terms_wide.csv"
wide_sig.to_csv(csv_sig, index=False)
print(f"Saved: {csv_sig}")

# Add simplified label + stars to the long table
all_long = all_long.copy()
all_long["term_simple"] = all_long["term"].apply(lambda s: mf.simplify_term(s, keep_prefix=False))
all_long["stars"] = all_long["p_boot"].apply(mf.stars)

csv_all_pretty = out_dir / "bootstrap_ols_all_llms_warmth_long_pretty.csv"
all_long.to_csv(csv_all_pretty, index=False)
print(f"Saved: {csv_all_pretty}")

wide_sig = wide_sig.copy()
wide_sig["term_simple"] = wide_sig["term"].apply(lambda s: mf.simplify_term(s, keep_prefix=False))

# Add stars per LLM based on p_boot_<llm>
for llm in ["llama", "qwen", "mistral"]:
    pcol = f"p_boot_{llm}"
    if pcol in wide_sig.columns:
        wide_sig[f"stars_{llm}"] = wide_sig[pcol].apply(mf.stars)

# (Optional) move term_simple next to term for readability
cols = ["term", "term_simple"] + [c for c in wide_sig.columns if c not in ("term", "term_simple")]
wide_sig = wide_sig[cols]

csv_sig_pretty = out_dir / "bootstrap_ols_warmth_score_significant_terms_wide_pretty.csv"
wide_sig.to_csv(csv_sig_pretty, index=False)
print(f"Saved: {csv_sig_pretty}")



# Model Diagnostics

In [ ]:
import statsmodels.formula.api as smf

rhs = "C(gender) + C(race) + C(religion) + C(transness) + C(concept)"
formula_resp = f"response_recoded ~ {rhs}"

def fit_ols_response(df, name):
    if df is None:
        raise ValueError(f"{name}: dataframe is None.")
    df_use = df.dropna(subset=["response_recoded"]).copy()
    if df_use.empty:
        raise ValueError(f"{name}: empty after dropping NA response_recoded.")
    return smf.ols(formula=formula_resp, data=df_use).fit()

model_resp_ols_llama   = fit_ols_response(df_llama_ready,   "model_resp_ols_llama")
model_resp_ols_mistral = fit_ols_response(df_mistral_ready, "model_resp_ols_mistral")
model_resp_ols_qwen    = fit_ols_response(df_qwen_ready,    "model_resp_ols_qwen")


In [ ]:
model_c_qwen_maihda = res_comp_qwen_maihda["model_1B"]
model_w_qwen_maihda = res_warmth_qwen_maihda["model_1B"]
model_c_mistral_maihda = res_comp_mistral_maihda["model_1B"]
model_w_mistral_maihda = res_warmth_mistral_maihda["model_1B"]
model_c_llama_maihda = res_comp_llama_maihda["model_1B"]
model_w_llama_maihda = res_warmth_llama_maihda["model_1B"]

all_models = {
    "all_llama_ols": (model_resp_ols_llama, df_llama_ready, "response_recoded", "ols"),
    "all_mistral_ols": (model_resp_ols_mistral, df_mistral_ready, "response_recoded", "ols"),
    "all_qwen_ols": (model_resp_ols_qwen, df_qwen_ready, "response_recoded", "ols"),
    "c_qwen":     (model_c_ols_qwen, df_qwen_ready, "competence_score", "ols"),
    "w_qwen":     (model_w_ols_qwen, df_qwen_ready, "warmth_score", "ols"),
    "c_mistral":  (model_c_ols_mistral, df_mistral_ready, "competence_score", "ols"),
    "w_mistral":  (model_w_ols_mistral, df_mistral_ready, "warmth_score", "ols"),
    "c_llama":    (model_c_ols_llama, df_llama_ready, "competence_score", "ols"),
    "w_llama":    (model_w_ols_llama, df_llama_ready, "warmth_score", "ols"),
    "comp_llama_maihda": (model_c_llama_maihda, df_llama_wc, "competence_score", "mai"),
    "warmth_llama_maihda": (model_w_llama_maihda,   df_llama_wc, "warmth_score", "mai"),
    "comp_mistral_maihda": (model_c_mistral_maihda, df_mistral_wc, "competence_score", "mai"),
    "warmth_mistral_maihda": (model_w_mistral_maihda, df_mistral_wc, "warmth_score", "mai"),
    "comp_qwen_maihda": (model_c_qwen_maihda, df_qwen_wc, "competence_score", "mai"),
    "warmth_qwen_maihda": (model_w_qwen_maihda, df_qwen_wc, "warmth_score", "mai"),
}

results = {}

for name, (model, df, yvar, modeltype) in all_models.items():
    if modeltype == "ols":
        diag = mf.OLSDiagnostics(model, df=df, response_var=yvar)
    elif modeltype == "mai":
        diag = mf.MAIHDADiagnostics(model, df=df, response_var=yvar)
    else:
        raise ValueError(f"Unknown model type: {modeltype}")

    print(f"\nRunning diagnostics for: {name}")
    results[name] = diag.report()

from pprint import pprint

for name, r in results.items():
    print("\n======================")
    print(name)
    print("======================")
    pprint(r)




In [ ]:
import pandas as pd

def extract_variance_components(model):
    tau2 = float(model.cov_re.iloc[0, 0])
    omega2 = float(model.scale)
    icc = tau2 / (tau2 + omega2)
    return tau2, omega2, icc

# Mapping: full model → dictionary where null & full live
maihda_sources = {
    "comp_llama_maihda": res_comp_llama_maihda,
    "warmth_llama_maihda": res_warmth_llama_maihda,
    "comp_mistral_maihda": res_comp_mistral_maihda,
    "warmth_mistral_maihda": res_warmth_mistral_maihda,
    "comp_qwen_maihda": res_comp_qwen_maihda,
    "warmth_qwen_maihda": res_warmth_qwen_maihda,
}

rows = []

for name, (full_model, df, yvar, modeltype) in all_models.items():
    if modeltype != "mai":
        continue
    
    # get the dict where model_1A and model_1B live
    source = maihda_sources[name]

    null_model = source["model_1A"]   # random intercept only
    full_model = source["model_1B"]   # random intercept + fixed effects

    # extract variances for full model
    tau2_full, omega2_full, icc_full = extract_variance_components(full_model)
    
    # extract tau^2 for null (PCV)
    tau2_null, _, _ = extract_variance_components(null_model)
    
    # compute PCV
    pcv = 100 * (tau2_null - tau2_full) / tau2_null

    rows.append({
        "Model": name,
        "ICC": icc_full,
        "tau2": tau2_full,
        "omega2": omega2_full,
        "VPC": icc_full,  # identical for 2-level model
        "PCV (%)": pcv
    })

df_results = pd.DataFrame(rows)
print(df_results)


# Plot in Warmth Competence-Space

## Plot Coefficient in Warmth Competence with CI 

In [ ]:
from importlib import reload
mf = reload(mf)

In [ ]:
mf.plot_2d_coefficients_with_bootstrap_ellipses(df_llama_wc, "Llama", save_dir = "plot_boot_wc_llama")
mf.plot_2d_coefficients_with_bootstrap_ellipses(df_qwen_wc, "Qwen", save_dir = "plot_boot_wc_qwen")
mf.plot_2d_coefficients_with_bootstrap_ellipses(df_mistral_wc, "Mistral", save_dir = "plot_boot_wc_mistral")

In [ ]:
print(res_warmth_llama_maihda.keys())
print(res_comp_llama_maihda.keys())
print(res_warmth_llama_maihda["fixed_effects"].head())
print(res_comp_llama_maihda["fixed_effects"].head())


# Plot MAIHDA only

In [ ]:
from importlib import reload
mf = reload(mf)


strata_all_llama, strata_labeled_llama = mf.plot_intersectional_bias_space(
    res_warmth_llama_maihda, 
    res_comp_llama_maihda,
    model_name="Llama",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=["Black_woman_not_mentioned_not_mentioned", "Black_man_not_mentioned_not_mentioned", "Black_woman_not_mentioned_trans", "not_mentioned_man_not_mentioned_trans"]
)


top_intersectional_llama = mf.identify_non_additive_groups(
    res_warmth_llama_maihda,
    res_comp_llama_maihda,
    top_n=15
)

strata_all_qwen, strata_labeled_qwen = mf.plot_intersectional_bias_space(
    res_warmth_qwen_maihda, 
    res_comp_qwen_maihda,
    model_name="Qwen",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=["Black_woman_not_mentioned_not_mentioned", "Black_man_not_mentioned_not_mentioned", "Black_woman_not_mentioned_trans", "not_mentioned_man_not_mentioned_trans"]
)

top_intersectional_qwen = mf.identify_non_additive_groups(
    res_warmth_qwen_maihda,
    res_comp_qwen_maihda,
    top_n=15
)

strata_all_mistral, strata_labeled_mistral = mf.plot_intersectional_bias_space(
    res_warmth_mistral_maihda, 
    res_comp_mistral_maihda,
    model_name="Mistral",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=["Black_woman_not_mentioned_not_mentioned", "Black_man_not_mentioned_not_mentioned", "Black_woman_not_mentioned_trans", "not_mentioned_man_not_mentioned_trans"]
)

# 3. Identify strongest non-additive effects
top_intersectional_mistral = mf.identify_non_additive_groups(
    res_warmth_mistral_maihda,
    res_comp_mistral_maihda,
    top_n=15
)


In [ ]:
# Identify all mixed strata (c-. w+ or c+ w-)
mask_llama = (
    ((strata_all_llama["warmth_centered"] > 0) & 
     (strata_all_llama["competence_centered"] < 0))
    |
    ((strata_all_llama["warmth_centered"] < 0) & 
     (strata_all_llama["competence_centered"] > 0))
)

opposite_quadrants_llama = strata_all_llama[mask_llama]
strata_list_llama = opposite_quadrants_llama["stratum"].unique()

# Identify all mixed strata (c-. w+ or c+ w-)
mask_qwen = (
    ((strata_all_qwen["warmth_centered"] > 0) & 
     (strata_all_qwen["competence_centered"] < 0))
    |
    ((strata_all_qwen["warmth_centered"] < 0) & 
     (strata_all_qwen["competence_centered"] > 0))
)

opposite_quadrants_qwen = strata_all_qwen[mask_qwen]
strata_list_qwen = opposite_quadrants_qwen["stratum"].unique()


# Identify all mixed strata (c-. w+ or c+ w-)
mask_mistral = (
    ((strata_all_mistral["warmth_centered"] > 0) & 
     (strata_all_mistral["competence_centered"] < 0))
    |
    ((strata_all_mistral["warmth_centered"] < 0) & 
     (strata_all_mistral["competence_centered"] > 0))
)

opposite_quadrants_mistral = strata_all_mistral[mask_mistral]
strata_list_mistral = opposite_quadrants_mistral["stratum"].unique()

In [ ]:

strata_all_llama, strata_labeled_llama = mf.plot_intersectional_bias_space(
    res_warmth_llama_maihda, 
    res_comp_llama_maihda,
    model_name="Llama",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=strata_list_llama
)


top_intersectional_llama = mf.identify_non_additive_groups(
    res_warmth_llama_maihda,
    res_comp_llama_maihda,
    top_n=15
)

strata_all_qwen, strata_labeled_qwen = mf.plot_intersectional_bias_space(
    res_warmth_qwen_maihda, 
    res_comp_qwen_maihda,
    model_name="Qwen",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=strata_list_qwen
)

top_intersectional_qwen = mf.identify_non_additive_groups(
    res_warmth_qwen_maihda,
    res_comp_qwen_maihda,
    top_n=15
)

strata_all_mistral, strata_labeled_mistral = mf.plot_intersectional_bias_space(
    res_warmth_mistral_maihda, 
    res_comp_mistral_maihda,
    model_name="Mistral",
    n_extremes=5,  # top/bottom 5 per dimension
    label_single_identity=True,
    additional_strata=strata_list_mistral
)

# 3. Identify strongest non-additive effects
top_intersectional_mistral = mf.identify_non_additive_groups(
    res_warmth_mistral_maihda,
    res_comp_mistral_maihda,
    top_n=15
)


## MAIHDA Bootstrapping

#### Llama Bootstrapping MAIHDA


In [ ]:
fe_boot, base_fit = mf.bootstrap_fixed_effects(
    df=df_llama_ready,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    B=2000,                 # für Probelauf kleiner, final 1500–5000
    alpha=0.05,
    mode="cluster",         # oder "parametric"
    reml=True,
    optimizer="powell",     # probier auch 'lbfgs'
    seed=123
)

# Export
#fe_boot.to_csv("maihda_bootstrap_output/fe_bootstrap2000_llama.csv", index=False)

In [ ]:
# Export
fe_boot.to_csv("maihda_bootstrap_output/fe_bootstrap_llama.csv", index=False)

#### Mistral Bootstrapping MAIHDA


In [ ]:
fe_boot, base_fit = mf.bootstrap_fixed_effects(
    df=df_mistral_ready,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    B=2000,                 # für Probelauf kleiner, final 1500–5000
    alpha=0.05,
    mode="cluster",         # oder "parametric"
    reml=True,
    optimizer="powell",     # probier auch 'lbfgs'
    seed=123
)

# Export
fe_boot.to_csv("maihda_bootstrap_output/fe_bootstrap2000_mistral.csv", index=False)

#### Qwen Bootstrapping MAIHDA


In [ ]:
fe_boot, base_fit = mf.bootstrap_fixed_effects(
    df=df_qwen_ready,
    response_var="response_recoded",
    fixed_effects=("race","gender","religion","transness","concept"),
    stratum_col="stratum",
    B=2000,                 # für Probelauf kleiner, final 1500–5000
    alpha=0.05,
    mode="cluster",         # oder "parametric"
    reml=True,
    optimizer="powell",     # probier auch 'lbfgs'
    seed=123
)

# Export
fe_boot.to_csv("maihda_bootstrap_output/fe_bootstrap2000_qwen.csv", index=False)

### Random Intercepts Bootstrap Analysis

In [ ]:
# Example: Bootstrap analysis for random intercepts in Llama MAIHDA model

# Bootstrap random intercepts only
random_intercepts_results, base_model = mf.bootstrap_random_intercepts(
    df=df_llama_ready,
    response_var="response_recoded",
    fixed_effects=("race", "gender", "religion", "transness", "concept"),
    stratum_col="stratum",
    B=1000,  # Use smaller B for demonstration
    alpha=0.05,
    mode="parametric",  # Recommended for random effects
    seed=123,
    verbose=True
)

# Save results
random_intercepts_results.to_excel("random_intercepts_bootstrap_llama.xlsx", index=False)

# Show top significant random intercepts
print("\n TOP 10 MOST SIGNIFICANT RANDOM INTERCEPTS:")
significant_only = random_intercepts_results[random_intercepts_results['significant']].copy()
if not significant_only.empty:
    top_significant = significant_only.head(10)
    for _, row in top_significant.iterrows():
        print(f"  {row['stratum'][:50]:50} {row['random_intercept']:8.4f} [{row['boot_ci_lower']:7.4f}, {row['boot_ci_upper']:7.4f}]")
else:
    print("  No significant random intercepts found.")

In [ ]:
# Comprehensive bootstrap analysis (all components)
bootstrap_results_complete = mf.bootstrap_maihda_complete(
    df=df_llama_ready,
    response_var="response_recoded",
    fixed_effects=("race", "gender", "religion", "transness", "concept"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_complete,
    output_dir="maihda_bootstrap_complete_output_llama",
    base_name="llama_maihda_bootstrap",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

In [ ]:
# Create visualization of random intercepts with bootstrap CIs
if 'random_intercepts' in bootstrap_results_complete and not bootstrap_results_complete['random_intercepts'].empty:
    fig = mf.plot_random_intercept_bootstrap(
        bootstrap_results_complete['random_intercepts'],
        n_top=15,
        n_bottom=15,
        figsize=(14, 12),
        title_suffix=" - Llama Model",
        show_insignificant=True
    )
    plt.show()
    
    # Additional analysis: Compare significant vs non-significant
    re_df = bootstrap_results_complete['random_intercepts']
    n_total = len(re_df)
    n_significant = re_df['significant'].sum()
    
    print(f"\n📊 RANDOM INTERCEPTS SUMMARY:")
    print(f"   Total strata: {n_total}")
    print(f"   Significant intercepts: {n_significant} ({100*n_significant/n_total:.1f}%)")
    print(f"   Range: [{re_df['random_intercept'].min():.4f}, {re_df['random_intercept'].max():.4f}]")
    print(f"   Mean bootstrap SE: {re_df['boot_se'].mean():.4f}")

## Bootstrapping for MAIHDA Warmth-Competence

## LLaMA


In [ ]:
# Comprehensive bootstrap analysis (all components)
bootstrap_results_llama_w = mf.bootstrap_maihda_complete(
    df=df_llama_wc,
    response_var="warmth_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_llama_w,
    output_dir="maihda_bootstrap_complete_output_llama_wc",
    base_name="llama_maihda_bootstrap_w",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

# Comprehensive bootstrap analysis (all components)
bootstrap_results_llama_c = mf.bootstrap_maihda_complete(
    df=df_llama_wc,
    response_var="competence_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_llama_w,
    output_dir="maihda_bootstrap_complete_output_llama_wc",
    base_name="llama_maihda_bootstrap_c",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

### Qwen

In [ ]:
# Comprehensive bootstrap analysis (all components)
bootstrap_results_qwen_w = mf.bootstrap_maihda_complete(
    df=df_qwen_wc,
    response_var="warmth_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_qwen_w,
    output_dir="maihda_bootstrap_complete_output_qwen_wc",
    base_name="qwen_maihda_bootstrap_w",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

# Comprehensive bootstrap analysis (all components)
bootstrap_results_qwen_c = mf.bootstrap_maihda_complete(
    df=df_qwen_wc,
    response_var="competence_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_qwen_c,
    output_dir="maihda_bootstrap_complete_output_qwen_wc",
    base_name="qwen_maihda_bootstrap_c",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

## Mistral

In [ ]:
# Comprehensive bootstrap analysis (all components)
bootstrap_results_mistral_w = mf.bootstrap_maihda_complete(
    df=df_mistral_wc,
    response_var="warmth_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_mistral_w,
    output_dir="maihda_bootstrap_complete_output_mistral_wc",
    base_name="mistral_maihda_bootstrap_w",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

# Comprehensive bootstrap analysis (all components)
bootstrap_results_mistral_c = mf.bootstrap_maihda_complete(
    df=df_mistral_wc,
    response_var="competence_score",
    fixed_effects=("race", "gender", "religion", "transness"),
    stratum_col="stratum",
    B_fixed=1000,         # Bootstrap samples for fixed effects
    B_random=1000,        # Bootstrap samples for random intercepts
    B_means=500,          # Bootstrap samples for stratum means
    alpha=0.05,
    mode_fixed="cluster", # Cluster bootstrap for fixed effects
    mode_random="parametric",  # Parametric bootstrap for random effects
    seed=123,
    verbose=True
)

# Export all results with visualizations
output_paths = mf.export_maihda_bootstrap_results(
    bootstrap_results_mistral_c,
    output_dir="maihda_bootstrap_complete_output_mistral_wc",
    base_name="mistral_maihda_bootstrap_c",
    create_plots=True,
    save_excel=True,
    save_csv=True
)

# Export Results

In [ ]:
EXCEL_OUT = "model_results_publication.xlsx"
TOP_N = 5


# ----------------------------------------------------------------------
# OLS extractor
# ----------------------------------------------------------------------
def extract_ols_table(name, model):
    tbl = model.summary2().tables[1].reset_index()
    tbl.rename(columns={"index": "term"}, inplace=True)
    
    # clean column names
    rename = {
        "Coef.": "estimate",
        "Std.Err.": "se",
        "P>|t|": "pvalue",
        "[0.025": "ci_low",
        "0.975]": "ci_high",
    }
    tbl = tbl.rename(columns={k: v for k, v in rename.items() if k in tbl.columns})
    
    tbl.insert(0, "model", name)
    tbl.insert(1, "effect_type", "fixed_ols")
    return tbl


# ----------------------------------------------------------------------
# MAIHDA fixed effects
# ----------------------------------------------------------------------
def extract_maihda_fixed(name, model):
    fe = pd.DataFrame({
        "term": model.fe_params.index,
        "estimate": model.fe_params.values,
        "se": model.bse_fe,
        "pvalue": model.pvalues.loc[model.fe_params.index],
    })
    
    ci = model.conf_int().loc[model.fe_params.index]
    fe["ci_low"] = ci[0].values
    fe["ci_high"] = ci[1].values
    
    fe.insert(0, "model", name)
    fe.insert(1, "effect_type", "fixed_maihda")
    return fe


# ----------------------------------------------------------------------
# MAIHDA random effects
# ----------------------------------------------------------------------
def extract_maihda_random(name, model, top_n=TOP_N):
    """
    Extract random intercepts from statsmodels MixedLM with correct SE.
    Uses model.random_effects_cov[group][0,0] to compute SE.
    """
    re_dict = model.random_effects
    cov_dict = model.random_effects_cov  # nested dict: group → DataFrame
    
    groups = []
    estimates = []
    ses = []
    
    for g, value in re_dict.items():
        est = float(value.iloc[0])
        var = cov_dict[g].iloc[0, 0]      # variance of the group-specific intercept
        se = np.sqrt(var)
        
        groups.append(g)
        estimates.append(est)
        ses.append(se)
    
    df = pd.DataFrame({"group": groups, "estimate": estimates, "se": ses})
    
    # compute CI
    df["ci_low"] = df["estimate"] - 1.96 * df["se"]
    df["ci_high"] = df["estimate"] + 1.96 * df["se"]
    
    # significance
    df["significant"] = (df["ci_low"] > 0) | (df["ci_high"] < 0)
    
    # extreme groups
    df["abs_est"] = df["estimate"].abs()
    top = df.nlargest(top_n, "abs_est")
    bottom = df.nsmallest(top_n, "abs_est")
    sig = df[df["significant"]]
    
    selected = pd.concat([sig, top, bottom]).drop_duplicates(subset=["group"])
    
    selected.insert(0, "model", name)
    selected.insert(1, "effect_type", "random_intercept")
    
    return selected.sort_values("estimate")

# ----------------------------------------------------------------------
# MAIN: write everything into Excel
# ----------------------------------------------------------------------
with pd.ExcelWriter(EXCEL_OUT, engine="openpyxl") as writer:
    for name, (model, df, yvar, modeltype) in all_models.items():
        print(f"Processing {name} ({modeltype})")
        
        if modeltype == "ols":
            tbl = extract_ols_table(name, model)
        
        elif modeltype == "mai":
            fe = extract_maihda_fixed(name, model)
            re = extract_maihda_random(name, model)
            tbl = pd.concat([fe, re], ignore_index=True)
        
        else:
            raise ValueError("Unknown model type")
        
        tbl.to_excel(writer, sheet_name=name[:31], index=False)

print(f"Excel written to {EXCEL_OUT}")


In [ ]:
def merge_random_effects(comp_df, warm_df):
    comp_df = comp_df.rename(columns={
        "estimate": "comp_est",
        "ci_low": "comp_ci_low",
        "ci_high": "comp_ci_high",
        "significant": "comp_sig"
    })
    warm_df = warm_df.rename(columns={
        "estimate": "warm_est",
        "ci_low": "warm_ci_low",
        "ci_high": "warm_ci_high",
        "significant": "warm_sig"
    })

    merged = pd.merge(comp_df, warm_df, on="group", how="outer")

    def alignment(row):
        if pd.isna(row["comp_est"]) or pd.isna(row["warm_est"]):
            return "NA"
        if row["comp_est"] > 0 and row["warm_est"] > 0:
            return "↑↑"
        if row["comp_est"] < 0 and row["warm_est"] < 0:
            return "↓↓"
        return "conflict"

    merged["alignment"] = merged.apply(alignment, axis=1)
    merged["delta_magnitude"] = (
        merged["comp_est"].abs() - merged["warm_est"].abs()
    ).abs()

    return merged.sort_values("alignment")



def group_maihda_pairs(all_models):
    """
    Returns a dict:
      'llama': ('comp_llama_maihda', 'warmth_llama_maihda')
      'mistral': (...)
      'qwen': (...)
    """
    pairs = {}

    for name in all_models:
        if name.endswith("_maihda"):
            prefix = name.replace("comp_", "").replace("warmth_", "").replace("_maihda", "")
            if prefix not in pairs:
                pairs[prefix] = [None, None]
            if name.startswith("comp_"):
                pairs[prefix][0] = name
            elif name.startswith("warmth_"):
                pairs[prefix][1] = name

    # clean into tuples
    out = {k: tuple(v) for k, v in pairs.items()}
    return out
maihda_pairs = group_maihda_pairs(all_models)

In [ ]:
with pd.ExcelWriter("maihda_comparison.xlsx", engine="openpyxl") as writer:
    
    # 1) write OLS models normally (you already had code)
    for name, (model, df, yvar, modeltype) in all_models.items():
        if modeltype == "ols":
            tbl = extract_ols_table(name, model)
            tbl.to_excel(writer, sheet_name=name[:31], index=False)

    # 2) MAIHDA — process in pairs automatically
    pairs = group_maihda_pairs(all_models)

    for llm, (comp_name, warm_name) in pairs.items():
        comp_model = all_models[comp_name][0]
        warm_model = all_models[warm_name][0]

        # fixed effects
        fe_comp = extract_maihda_fixed(comp_name, comp_model)
        fe_warm = extract_maihda_fixed(warm_name, warm_model)

        # random effects
        re_comp = extract_maihda_random(comp_name, comp_model)
        re_warm = extract_maihda_random(warm_name, warm_model)

        # merged comparison
        re_compare = merge_random_effects(re_comp, re_warm)

        # write sheets
        fe_combined = pd.merge(
            fe_comp, fe_warm,
            on="term", how="outer", suffixes=("_comp", "_warm")
        )
        fe_combined.to_excel(writer, sheet_name=f"{llm}_fixed", index=False)
        re_compare.to_excel(writer, sheet_name=f"{llm}_random_compare", index=False)


In [ ]:
pivot = df_mistral_ready.pivot_table(values='response', index='vignette_id', columns='concept')
correlation = pivot['ADM'].corr(pivot['CON'])
df_mistral_ready.groupby('concept')['response'].describe()

In [ ]:
# Install adjustText for improved label positioning (if not already installed)
try:
    from adjustText import adjust_text
    print("✅ adjustText already installed")
except ImportError:
    print("📦 Installing adjustText...")
    import sys
    !{sys.executable} -m pip install adjustText
    print("✅ adjustText installed successfully")

# Bootstrap by Resampling

In [ ]:
mf = reload(mf)

In [ ]:
df_llama = df_llama.rename(columns={"gender_alignment":"transness"})  
cols = ["race", "gender", "religion", "transness"]
df_llama[cols] = df_llama[cols].fillna("not_mentioned")
res_all_llama = mf.bootstrap_all_models_wrapper(
     df_raw=df_llama,
    fixed_effects=("race","gender","religion","transness"),
    B=1000,
    alpha=0.05,
    seed=42,
    verbose=True,
    save_dir="results/llama",
    save_prefix="llama_bootstrap",
    save_excel=True,
    save_csv=False
)



In [ ]:
df_mistral = df_mistral.rename(columns={"gender_alignment":"transness"})  
cols = ["race", "gender", "religion", "transness"]
df_mistral[cols] = df_mistral[cols].fillna("not_mentioned")
res_all_mistral = mf.bootstrap_all_models_wrapper(
    df_raw=df_mistral,
    fixed_effects=("race","gender","religion","transness"),
    B=1000,
    alpha=0.05,
    seed=42,
    verbose=True,
    save_dir="results/mistral",
    save_prefix="mistral_bootstrap",
    save_excel=True,
    save_csv=False
)


df_qwen = df_qwen.rename(columns={"gender_alignment":"transness"})  
cols = ["race", "gender", "religion", "transness"]
df_qwen[cols] = df_qwen[cols].fillna("not_mentioned")
res_all_qwen = mf.bootstrap_all_models_wrapper(
    df_raw=df_qwen,
    fixed_effects=("race","gender","religion","transness"),
    B=1000,
    alpha=0.05,
    seed=42,
    verbose=True,
    save_dir="results/qwen",
    save_prefix="qwen_bootstrap",
    save_excel=True,
    save_csv=False
)




# Export

In [ ]:
mf = reload(mf)

In [ ]:
ols_warmth_master = mf.build_master_table(
    res_all_llama["ols_warmth"],
    res_all_qwen["ols_warmth"],
    res_all_mistral["ols_warmth"],
    key="ols"
)

ols_comp_master = mf.build_master_table(
    res_all_llama["ols_competence"],
    res_all_qwen["ols_competence"],
    res_all_mistral["ols_competence"],
    key="ols"
)


In [ ]:
maihda_warmth_master = mf.build_master_table(
    res_all_llama["maihda_warmth"]["result"],
    res_all_qwen["maihda_warmth"]["result"],
    res_all_mistral["maihda_warmth"]["result"],
    key="fixed_effects"
)

maihda_comp_master = mf.build_master_table(
    res_all_llama["maihda_competence"]["result"],
    res_all_qwen["maihda_competence"]["result"],
    res_all_mistral["maihda_competence"]["result"],
    key="fixed_effects"
)

In [ ]:
re_with_text = mf.attach_vignette_text_all(res_all_llama, res_all_qwen, res_all_mistral, vignette_complete)
from pathlib import Path

out_dir = Path("random_intercepts_csv")
out_dir.mkdir(exist_ok=True)

for name, df in re_with_text.items():
    path = out_dir / f"{name}.csv"
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


In [ ]:
export_dir = Path("exports")
export_dir.mkdir(exist_ok=True)

ols_warmth_master.to_csv(export_dir / "ols_warmth_master.csv", index=False)
ols_comp_master.to_csv(export_dir / "ols_comp_master.csv", index=False)
maihda_warmth_master.to_csv(export_dir / "maihda_warmth_master.csv", index=False)
maihda_comp_master.to_csv(export_dir / "maihda_comp_master.csv", index=False)

print("✅ Exported master tables to", export_dir.resolve())

# Save and Load results data for further processing

In [ ]:
import pickle
from pathlib import Path

# Create directory for saved results
save_dir = Path("saved_results")
save_dir.mkdir(exist_ok=True)

# Save all three result objects
with open(save_dir / "res_all_llama.pkl", "wb") as f:
    pickle.dump(res_all_llama, f)
    
with open(save_dir / "res_all_qwen.pkl", "wb") as f:
    pickle.dump(res_all_qwen, f)
    
with open(save_dir / "res_all_mistral.pkl", "wb") as f:
    pickle.dump(res_all_mistral, f)

print("✅ All results saved successfully!")
print(f"   Location: {save_dir.resolve()}")
print(f"   Files: res_all_llama.pkl, res_all_qwen.pkl, res_all_mistral.pkl")

In [ ]:
import pickle
from pathlib import Path

# Load all three result objects
save_dir = Path("saved_results")

with open(save_dir / "res_all_llama.pkl", "rb") as f:
    res_all_llama = pickle.load(f)
    
with open(save_dir / "res_all_qwen.pkl", "rb") as f:
    res_all_qwen = pickle.load(f)
    
with open(save_dir / "res_all_mistral.pkl", "rb") as f:
    res_all_mistral = pickle.load(f)

print("✅ All results loaded successfully!")
print(f"\nres_all_llama keys: {list(res_all_llama.keys())}")
print(f"res_all_qwen keys: {list(res_all_qwen.keys())}")
print(f"res_all_mistral keys: {list(res_all_mistral.keys())}")

### MAIHDA Graph

In [ ]:
for name, res in [
    ("LLaMA", res_all_llama),
    ("Qwen", res_all_qwen),
    ("Mistral", res_all_mistral),
]:
    mf.plot_maihda_fixed_and_strata_adjusttext_fixedFE(
        res_all=res,
        llm_name=name,
        vignette_complete=vignette_complete,
        n_extremes=5, 
        show_fixed_effects=False
    )
    plt.show()
